# Pipelines DVC et compression de modèles

In [ ]:
# @title Mise en place de l'environnement
!rm -rf sample_data .config
!git config --global user.email "jane@doe.eu"
!git config --global user.name "Jane Doe"
!git config --global init.defaultBranch main
!pip install dvc dvclive --quiet
!pip install -U uv

## Mise en place des dépôts DVC & Git

Initialisez DVC & Git.

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!git init

In [ ]:
!dvc init

In [ ]:
!git commit -m "Initialisation de DVC et Git"

## Ajout des données à DVC

Avec [`dvc import-url`](https://dvc.org/doc/command-reference/import-url), téléchargez l'archive zip suivante, que nous allons utiliser :

    https://github.com/shuuchuu/dataset-landscape/archive/refs/heads/main.zip

Utilisez `data.zip` comme nom de sortie.

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

Commitez les modifications dans `git`.

### Solution

In [ ]:
!dvc import-url https://github.com/shuuchuu/dataset-landscape/archive/refs/heads/main.zip data.zip

In [ ]:
!git add .gitignore data.zip.dvc

In [ ]:
!git commit -m "Ajout des données"

## Création d'une étape de pipeline pour extraire le contenu de l'archive zip

Avec `dvc stage add`, ou en éditant le fichier `dvc.yaml`, créez une étape de pipeline dvc qui extrait les fichiers de `data.zip` (vous pouvez utiliser `unzip` pour cela).

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

### Solution

In [ ]:
!dvc stage add -n decompress -d data.zip -o dataset-landscape-main unzip data.zip

In [ ]:
!dvc repro

In [ ]:
!git add dvc.lock dvc.yaml .gitignore

In [ ]:
!git commit -m "Ajout de l'étape d'extraction"

## Mise en place d'un projet Python

Mettez en place un projet ou un fichier Python pour pouvoir exécuter facilement votre code source.

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

### Solution

1. Lancez `uv init --package --name compression .` pour créer `pyproject.toml`
2. Créez le dossier source et un `__init__.py` dedans
3. Ajoutez un point d'entrée :
        [project.scripts]
        commandname = "folder.file:function"
4. Lancez `uv sync`
6. `!commandname`

In [ ]:
!uv init --package --name compression .

In [ ]:
%%writefile src/compression/__init__.py
def main() -> None:
    print("Hello World")

In [ ]:
!uv sync

In [ ]:
!uv run compression

## Préparation des données

Pour préparer les données avant d'entraîner et de compresser le modèle, nous allons utiliser la fonction suivante, que vous pouvez intégrer à votre code comme bon vous semble :

In [ ]:
# @title Code de préparation des données
import pathlib

import cv2
import numpy
import sklearn.utils

CLASS_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
CLASS_INDICES = {l: i for i, l in enumerate(CLASS_NAMES)}


def get_images(
  dir_path: pathlib.Path, image_size: tuple[int, int], shuffle: bool = True
) -> tuple[numpy.ndarray, numpy.ndarray, numpy.ndarray]:
  images = []
  labels = []
  file_paths = []

  for subdir_path in dir_path.iterdir():
    label = CLASS_INDICES.get(subdir_path.name)

    for image_path in subdir_path.iterdir():
      image = cv2.imread(str(image_path))
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
      image = cv2.resize(image, image_size)
      image = image.astype("float32")
      images.append(image)
      labels.append(label)
      file_paths.append(image_path)
  images_array = numpy.array(images)
  labels_array = numpy.array(labels)
  file_paths_array = numpy.array(file_paths)

  if shuffle:
    images_array, labels_array, file_paths_array = sklearn.utils.shuffle(
      images_array, labels_array, file_paths_array
    )
  return images_array, labels_array, file_paths_array

## Entraînement d'un modèle simple

Nous allons maintenant entraîner un modèle de classification simple (et historique qui plus est) : LeNet. Vous pouvez intégrer le code ci-dessous à votre projet comme bon vous semble.

In [ ]:
# @title Code de définition du modèle
import tensorflow as tf


def get_lenet(
  image_size: tuple[int, int], learning_rate: float = 1e-4
) -> tf.keras.models.Model:
  def conv(filters: int, padding: str) -> tf.keras.layers.Conv2D:
    return tf.keras.layers.Conv2D(
      filters=filters, kernel_size=5, padding=padding, activation="sigmoid"
    )

  def pooling() -> tf.keras.layers.MaxPooling2D:
    return tf.keras.layers.MaxPooling2D()

  def dense(units: int, activation: str = "sigmoid") -> tf.keras.layers.Dense:
    return tf.keras.layers.Dense(units, activation=activation)

  model = tf.keras.Sequential(
    [
      tf.keras.layers.InputLayer(input_shape=(*image_size, 3)),
      conv(6, "same"),
      pooling(),
      conv(16, "valid"),
      pooling(),
      tf.keras.layers.Flatten(),
      dense(120),
      dense(84),
      dense(6, activation="softmax"),
    ],
    name="le_net",
  )

  model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )

  return model

In [ ]:
# @title Code d'entraînement du modèle
from pathlib import Path

from ... import (
  get_images,  # À adapter selon l'organisation de votre code
  get_lenet,  # À adapter selon l'organisation de votre code
)


def train(
  data_dir: str,
  image_size: tuple[int, int],
  learning_rate: float,
) -> None:
  images, labels, paths = get_images(Path(data_dir), image_size)
  model = get_lenet(image_size, learning_rate)
  model.fit(images, labels, 128, epochs=3)
  model.save("landscape_classifier.keras")

### Définition de l'étape de pipeline

Vous pouvez maintenant définir une étape de pipeline pour entraîner un modèle.

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

In [ ]:
!dvc repro

## Compression du modèle après entraînement

Vous pouvez suivre ce [guide](https://www.tensorflow.org/model_optimization/guide/quantization/post_training) pour ajouter une étape de compression à votre entraînement. Vérifiez les performances du modèle après quantification.

In [ ]:
!# Votre commande ici, notez le ! qui préfixe les commandes bash dans Colab

In [ ]:
!dvc repro

## Solution

https://github.com/shuuchuu/compression